In [1]:
import json
import os
import re
from glob import glob

import pandas as pd

## 文件重命名

In [2]:
def get_foldername(root="."):
    folders = []
    for name in os.listdir(root):
        if os.path.isdir(name):
            print(name)
            folders.append(name)
    return folders


def filename_cleaner(folder: str) -> str:
    """清理文件名：替换特殊字符、去除冗余内容"""
    files = glob(f"{folder}/*")
    for filepath in sorted(files):
        dir = os.path.dirname(filepath)
        filename = os.path.basename(filepath)

        # print(filename)
        cleaned = re.sub("\\s+", "-", filename)
        cleaned = re.sub("\\+", "-", cleaned)
        cleaned = re.sub("_", "-", cleaned)
        cleaned = re.sub("--", "-", cleaned)
        cleaned = re.sub("\\(\\d+\\)", "", cleaned)
        cleaned = re.sub("(css|md)\\.", ".", cleaned)
        cleaned = re.sub("\\.\\.", ".", cleaned)
        cleaned = re.sub("-\\.", ".", cleaned)
        os.rename(filepath, f"{dir}/{cleaned}")


folders = get_foldername()
filename_cleaner(folders[0])

计科B241-7


## 读分

In [3]:
if folders[0].startswith("计科"):
    score_file = "2026-ml-成绩.xlsx"
elif folders[0].startswith("物联网"):
    score_file = "2026-cv-成绩.xlsx"
sheet_n = folders[0].split("-")[0]

n_theo, n_exp = 16, 7
col_names = (
    ["索引", "排名", "学号", "姓名"]
    + ["课堂"] * n_theo
    + ["成绩"]
    + [f"作业{i}" for i in range(1, 1 + n_exp)]
    + ["其他"] * 3
)

table_df = pd.read_excel(score_file, sheet_name=sheet_n, skiprows=4).reset_index()
table_df.columns = col_names
table_df.head()

,索引,排名,学号,姓名,课堂,课堂,课堂,课堂,课堂,课堂,...,作业1,作业2,作业3,作业4,作业5,作业6,作业7,其他,其他,其他
0,0,61.0,2.317010e+09,阿飞法,80.0,80.0,80.0,80.0,80.0,80.0,...,80.0,0.0,0.0,70.0,85.0,80.0,NaN,45.000000,NaN,34.000000
1,1,65.0,2.317010e+09,阿柯代·艾克拜,80.0,80.0,80.0,80.0,80.0,80.0,...,90.0,0.0,0.0,70.0,85.0,0.0,NaN,35.000000,NaN,30.000000
2,2,40.0,2.317010e+09,穆妮热·斯坎旦,80.0,80.0,80.0,80.0,80.0,80.0,...,80.0,85.0,100.0,90.0,0.0,80.0,NaN,62.142857,NaN,40.857143
3,3,48.0,2.317010e+09,伊尔潘·阿布都许克尔,80.0,80.0,80.0,80.0,80.0,80.0,...,80.0,80.0,95.0,0.0,85.0,80.0,NaN,60.000000,NaN,40.000000
4,4,60.0,2.317010e+09,阿丽米热·克依木,80.0,80.0,80.0,80.0,80.0,80.0,...,80.0,85.0,0.0,0.0,80.0,80.0,NaN,46.428571,NaN,34.571429


## 文件内检测

In [6]:
def grade(filepath: str, goals: dict, penalties: dict, init_score: int = 100):
    """批改单个文件"""
    print(f"Grading {filepath}...")
    with open(filepath, encoding="utf-8") as f:
        nb = json.load(f)

    code = ""
    for cell in nb.get("cells", []):
        if cell.get("cell_type") == "code":
            src = cell.get("source", [])
            code += "".join(src) if isinstance(src, list) else src

    violations = 0
    for pattern, penalty in penalties.items():
        if pattern in code:
            print(f"Violation found: '{pattern}'")
            violations += penalty

    for pattern, goal in goals.items():
        if pattern not in code:
            print(f"Goal not found: '{pattern}'")
            violations += goal

    final_score = max(0, init_score - violations)
    filename = os.path.basename(filepath)
    count = filename.count("-")

    if count == 2:
        os.rename(filepath, filepath.replace(".ipynb", f"-{final_score}.ipynb"))
    elif count == 3:
        score = filename.split("-")[-1]
        os.rename(filepath, filepath.replace(score, f"{final_score}.ipynb"))


# 配置
TARGET_DIR = folders[0]
RULES_FILE = "rules.json"
# 加载规则
if folders[0].startswith("计科"):
    rule = f"ipynb+m{folders[0][-1]}"
elif folders[0].startswith("物联网"):
    rule = "ipynb+v"

with open(RULES_FILE, encoding="utf-8") as f:
    goals = json.load(f).get(rule)
with open(RULES_FILE, encoding="utf-8") as f:
    penalties = json.load(f).get("ipynb-")

# 查找文件
files = glob(f"{TARGET_DIR}/*.ipynb")
# 批量批改
for f in files:
    grade(f, goals, penalties)

Grading 计科B241-7/07-2401040012-徐子涵-80.ipynb...
Violation found: 'warnings.filterwarnings'
Violation found: '
 '
Violation found: '
  '
Violation found: '
   '
Grading 计科B241-7/07-2401010016-郭嘉铖-75.ipynb...
Violation found: 'warnings.filterwarnings'
Violation found: 'plt.tight_layout'
Violation found: '
 '
Violation found: '
  '
Violation found: '
   '
Grading 计科B241-7/07-2401040041-莫瑞红-80.ipynb...
Violation found: 'warnings.filterwarnings'
Violation found: '
 '
Violation found: '
  '
Violation found: '
   '
Grading 计科B241-7/07-2401010048-呼奕彤-85.ipynb...
Violation found: '
 '
Violation found: '
  '
Violation found: '
   '
Grading 计科B241-7/07-2401010009-段鑫-85.ipynb...
Violation found: '
 '
Violation found: '
  '
Violation found: '
   '
Grading 计科B241-7/07-2401010039-夏诗琪-80.ipynb...
Violation found: 'warnings.filterwarnings'
Violation found: '
 '
Violation found: '
  '
Violation found: '
   '
Grading 计科B241-7/07-2401010022-左世龙-75.ipynb...
Violation found: 'warnings.filterwarnings'
Violati

## 登分

In [7]:
def score_extracter(folder, ext="ipynb"):
    files = glob(f"{folder}/*.{ext}")
    score_df = pd.DataFrame({"task_inds": [], "ids": [], "names": [], "scores": []})
    for f in files:
        filename = os.path.basename(f)
        # print(filename)
        filename_without_ext, _ = os.path.splitext(filename)
        items = filename_without_ext.split("-")
        new_row = pd.DataFrame(
            {
                "task_inds": [items[0]],
                "ids": [items[1]],
                "names": [items[2]],
                "scores": [items[3]],
            }
        )
        score_df = pd.concat([score_df, new_row], ignore_index=True)
        score_df.ids.astype("str")
    return score_df


score_df = score_extracter(TARGET_DIR)


def score_filler(score_df, sheet_n, col_names):
    table_df = pd.read_excel(score_file, sheet_name=sheet_n, skiprows=4).reset_index()
    table_df.columns = col_names
    for _, obj_row in score_df.iterrows():
        obj_task = obj_row["task_inds"]
        obj_id = obj_row["ids"]
        # obj_name = obj_row["names"]
        obj_score = obj_row["scores"]
        table_df.loc[table_df["学号"] == int(obj_id), f"作业{obj_task[1]}"] = int(
            obj_score
        )

    with pd.ExcelWriter(score_file, mode="a", if_sheet_exists="overlay") as writer:
        table_df.to_excel(writer, sheet_name=f"{sheet_n}-new")


score_filler(score_df, sheet_n, col_names)